In [1]:
#Import the required libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# -----------------------------
# 1. Custom Dataset
# -----------------------------
class TicketDataset(Dataset):
    def __init__(self, texts, labels_intent, labels_sentiment, tokenizer, max_len=128):
        self.texts = texts
        self.labels_intent = labels_intent
        self.labels_sentiment = labels_sentiment
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(text,
                             padding="max_length",
                             truncation=True,
                             max_length=self.max_len,
                             return_tensors="pt")
        item = {key: val.squeeze(0) for key, val in enc.items()}
        item["labels_intent"] = torch.tensor(self.labels_intent[idx], dtype=torch.long)
        item["labels_sentiment"] = torch.tensor(self.labels_sentiment[idx], dtype=torch.long)
        return item


In [3]:
# -----------------------------
# 2. Load Data from CSV (string labels)
# -----------------------------
df = pd.read_csv(r"D:\VS_CODE\INTEL-AIML\bank_nlp\files\01_support_tickets.csv")
df.head()

,ticket_id,date_created,customer_id,channel,category,sub_category,query_text,sentiment,risk_level,resolution_text,resolution_time_minutes,resolved_by,customer_satisfaction,escalated
0,TKT-10001,13-03-2023,CUST872246,Web Chat,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,I see a transaction of ₹999 I didn't make on 2...,Urgent,High,Unauthorized transaction reported on account. ...,28,Sneha R.,2,Yes
1,TKT-10002,05-11-2023,CUST127824,Branch,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,I see a transaction of ₹500 I didn't make on 1...,Neutral,High,Unauthorized transaction reported on account. ...,27,Auto-Resolved,5,No
2,TKT-10003,05-08-2023,CUST456778,Phone,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,"I see a transaction of ₹25,000 I didn't make o...",Neutral,High,Unauthorized transaction reported on account. ...,11,Anjali K.,2,No
3,TKT-10004,23-01-2023,CUST865179,Email,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,I see a transaction of ₹999 I didn't make on 1...,Confused,High,Unauthorized transaction reported on account. ...,8,Vikram P.,2,Yes
4,TKT-10005,05-12-2023,CUST338968,Phone,Fraud/Unauthorized,I see a transaction of ₹{amt} I didn't m,"I see a transaction of ₹25,000 I didn't make o...",Frustrated,High,Unauthorized transaction reported on account. ...,12,Priya S.,5,No


In [4]:
#Display the df shape
print("The df (tickets.csv ) shape is ",df.shape)
print("\nThe df columns are",df.columns)

The df (tickets.csv ) shape is  (200, 14)

The df columns are Index(['ticket_id', 'date_created', 'customer_id', 'channel', 'category',
       'sub_category', 'query_text', 'sentiment', 'risk_level',
       'resolution_text', 'resolution_time_minutes', 'resolved_by',
       'customer_satisfaction', 'escalated'],
      dtype='object')


In [5]:
#Rename the columns
df=df.rename(columns={'category':'intent','query_text':'text'})
print("\nThe df columns are",df.columns)


The df columns are Index(['ticket_id', 'date_created', 'customer_id', 'channel', 'intent',
       'sub_category', 'text', 'sentiment', 'risk_level', 'resolution_text',
       'resolution_time_minutes', 'resolved_by', 'customer_satisfaction',
       'escalated'],
      dtype='object')


In [6]:
#Check the unique values and their counts of intent column
df['intent'].value_counts()

intent
Fraud/Unauthorized    50
Loan                  50
KYC                   50
Account Access        50
Name: count, dtype: int64

In [7]:
##Check the unique values and their counts of sentiment column
df['sentiment'].value_counts()

sentiment
Anxious       39
Confused      37
Urgent        33
Neutral       32
Frustrated    31
Angry         28
Name: count, dtype: int64

In [8]:
# -----------------------------
# 3. DATA CLEANING
# -----------------------------

import re

def clean_text(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Replace ₹ with 'rs'
    text = text.replace("₹", "rs")
    
    # 3. Normalize dates like 25-apr-2023 → <DATE>
    text = re.sub(r"\d{1,2}-[a-z]{3}-\d{4}", "<DATE>", text)
    
    # 4. Normalize numbers → <NUM>
    text = re.sub(r"\d+", "<NUM>", text)
    
    # 5. Remove special characters (keep <DATE> and <NUM>)
    text = re.sub(r"[^a-z0-9\s<DATE><NUM>]", "", text)
    
    # 6. Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text.strip()

# Apply cleaning to your DataFrame column
df["text"] = df["text"].apply(clean_text)

# Quick check
print(df["text"].head())


0    i see a transaction of rs<NUM> i didnt make on...
1    i see a transaction of rs<NUM> i didnt make on...
2    i see a transaction of rs<NUM><NUM> i didnt ma...
3    i see a transaction of rs<NUM> i didnt make on...
4    i see a transaction of rs<NUM><NUM> i didnt ma...
Name: text, dtype: object


In [9]:
#Drop duplicates in text column 
df = df.drop_duplicates(subset="text", keep="last")


In [10]:

# Define mappings
intent_map = {"Fraud/Unauthorized": 0, "Loan": 1, "KYC": 2, "Account Access": 3}
sentiment_map = {"Anxious": 0, "Confused": 1, "Urgent": 2,"Neutral":3,"Frustrated":4,"Angry":5}

# Convert string labels to integers
df["intent"] = df["intent"].map(intent_map)
df["sentiment"] = df["sentiment"].map(sentiment_map)


#Split the df into train and test df
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

#Prepare the date for train_dataset
texts = train_df["text"].tolist()
labels_intent = train_df["intent"].tolist()
labels_sentiment = train_df["sentiment"].tolist()





In [11]:
print("Texts:", len(texts))
print("Intent labels:", len(labels_intent))
print("Sentiment labels:", len(labels_sentiment))


Texts: 37
Intent labels: 37
Sentiment labels: 37


In [12]:
print(df["intent"].unique())
print(df["sentiment"].unique())


[0 1 2 3]
[1 4 2 3 5 0]


In [13]:
# -----------------------------
# 4. Encoder + Heads
# -----------------------------
encoder = AutoModel.from_pretrained("distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

hidden_size = encoder.config.hidden_size
intent_head = nn.Linear(hidden_size, 4)      # 4 intents
sentiment_head = nn.Linear(hidden_size, 6)   # 6 sentiments

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) +
                       list(intent_head.parameters()) +
                       list(sentiment_head.parameters()), lr=5e-5)


# Train DataLoader
train_dataset = TicketDataset(texts, labels_intent, labels_sentiment, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)


from transformers import get_linear_schedule_with_warmup

epochs = 10
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_loader)*2,
    num_training_steps=len(train_loader) * epochs
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# -----------------------------
# 5. Training Loop
# -----------------------------
epochs=10
for epoch in range(epochs):
    for batch in train_loader:
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels_intent = batch["labels_intent"]
        labels_sentiment = batch["labels_sentiment"]

        outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0]

        logits_intent = intent_head(cls_repr)
        logits_sentiment = sentiment_head(cls_repr)

        loss_intent = loss_fn(logits_intent, labels_intent)
        loss_sentiment = loss_fn(logits_sentiment, labels_sentiment)
        loss = 0.5 * loss_intent + 0.5 * loss_sentiment

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
    scheduler.step()

    print(f"Epoch {epoch}, Loss {loss.item():.4f}")


Epoch 0, Loss 1.6824
Epoch 1, Loss 1.4949
Epoch 2, Loss 1.5269
Epoch 3, Loss 1.3722
Epoch 4, Loss 1.5270
Epoch 5, Loss 1.4379
Epoch 6, Loss 0.7826
Epoch 7, Loss 0.7617
Epoch 8, Loss 0.5274
Epoch 9, Loss 0.2396


In [15]:
#Prepare the data for Test Dataset
texts = test_df["text"].tolist()
labels_intent = test_df["intent"].tolist()
labels_sentiment = test_df["sentiment"].tolist()

# Test DataLoader
test_dataset = TicketDataset(texts, labels_intent, labels_sentiment, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=8)



In [16]:
#Evaluate the model for Accuracy
def evaluate(model, test_loader, intent_head, sentiment_head, device="cpu"):
    model.eval()
    correct_intent, correct_sentiment = 0, 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_intent = batch["labels_intent"].to(device)
            labels_sentiment = batch["labels_sentiment"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            cls_repr = outputs.last_hidden_state[:, 0]

            logits_intent = intent_head(cls_repr)
            logits_sentiment = sentiment_head(cls_repr)

            # Predictions
            preds_intent = torch.argmax(logits_intent, dim=1)
            preds_sentiment = torch.argmax(logits_sentiment, dim=1)

            # Accuracy counts
            correct_intent += (preds_intent == labels_intent).sum().item()
            correct_sentiment += (preds_sentiment == labels_sentiment).sum().item()
            total += labels_intent.size(0)

    acc_intent = correct_intent / total
    acc_sentiment = correct_sentiment / total
    print("Test Results :")
    print(f"Intent Accuracy: {acc_intent:.4f}, Sentiment Accuracy: {acc_sentiment:.4f}")
    return acc_intent, acc_sentiment


In [17]:
acc_intent, acc_sentiment=evaluate(encoder, test_loader, intent_head, sentiment_head)


Test Results :
Intent Accuracy: 0.6000, Sentiment Accuracy: 0.2000


In [18]:
# After training
encoder.save_pretrained("encoder_model_final_1")
tokenizer.save_pretrained("tokenizer_model_final_1")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('tokenizer_model_final_1\\tokenizer_config.json',
 'tokenizer_model_final_1\\tokenizer.json')

In [ ]:


# Save heads separately
torch.save(intent_head.state_dict(), "intent_head_final.pt")
torch.save(sentiment_head.state_dict(), "sentiment_head_final.pt")




'\n# Reload in Gradio\nintent_head.load_state_dict(torch.load("intent_head_final.pt"))\nsentiment_head.load_state_dict(torch.load("sentiment_head_final.pt"))\n'

In [ ]:
# -----------------------------
#  Inference
# -----------------------------


test_text = "Multiple small transactions are showing up that I didn't make"
enc = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)
outputs = encoder(**enc)
cls_repr = outputs.last_hidden_state[:, 0]
pred_intent = torch.argmax(intent_head(cls_repr), dim=1)
pred_sentiment = torch.argmax(sentiment_head(cls_repr), dim=1)

# Reverse mapping for readability
intent_inv_map = {v: k for k, v in intent_map.items()}
sentiment_inv_map = {v: k for k, v in sentiment_map.items()}

print("Predicted intent:", intent_inv_map[pred_intent.item()])
print("Predicted sentiment:", sentiment_inv_map[pred_sentiment.item()])


Predicted intent: Fraud/Unauthorized
Predicted sentiment: Neutral


In [ ]:

test_text = "Why is my account restricted? I submitted KYC documents last week."
enc = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)
outputs = encoder(**enc)
cls_repr = outputs.last_hidden_state[:, 0]
pred_intent = torch.argmax(intent_head(cls_repr), dim=1)
pred_sentiment = torch.argmax(sentiment_head(cls_repr), dim=1)

# Reverse mapping for readability
intent_inv_map = {v: k for k, v in intent_map.items()}
sentiment_inv_map = {v: k for k, v in sentiment_map.items()}

print("Predicted intent:", intent_inv_map[pred_intent.item()])
print("Predicted sentiment:", sentiment_inv_map[pred_sentiment.item()])


Predicted intent: KYC
Predicted sentiment: Angry


In [22]:


test_text = "My Aadhaar name and bank name don't match. How do I fix this"
enc = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)
outputs = encoder(**enc)
cls_repr = outputs.last_hidden_state[:, 0]
pred_intent = torch.argmax(intent_head(cls_repr), dim=1)
pred_sentiment = torch.argmax(sentiment_head(cls_repr), dim=1)

# Reverse mapping for readability
intent_inv_map = {v: k for k, v in intent_map.items()}
sentiment_inv_map = {v: k for k, v in sentiment_map.items()}

print("Predicted intent:", intent_inv_map[pred_intent.item()])
print("Predicted sentiment:", sentiment_inv_map[pred_sentiment.item()])


Predicted intent: KYC
Predicted sentiment: Confused


In [ ]:
test_text = "I'm locked out of my net banking. How do I regain access?"
enc = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)
outputs = encoder(**enc)
cls_repr = outputs.last_hidden_state[:, 0]
pred_intent = torch.argmax(intent_head(cls_repr), dim=1)
pred_sentiment = torch.argmax(sentiment_head(cls_repr), dim=1)

# Reverse mapping for readability
intent_inv_map = {v: k for k, v in intent_map.items()}
sentiment_inv_map = {v: k for k, v in sentiment_map.items()}

print("Predicted intent:", intent_inv_map[pred_intent.item()])
print("Predicted sentiment:", sentiment_inv_map[pred_sentiment.item()])


Predicted intent: Account Access
Predicted sentiment: Anxious


In [24]:
    
test_text = "Can I open a joint account with my spouse?"
enc = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)
outputs = encoder(**enc)
cls_repr = outputs.last_hidden_state[:, 0]
pred_intent = torch.argmax(intent_head(cls_repr), dim=1)
pred_sentiment = torch.argmax(sentiment_head(cls_repr), dim=1)

# Reverse mapping for readability
intent_inv_map = {v: k for k, v in intent_map.items()}
sentiment_inv_map = {v: k for k, v in sentiment_map.items()}

print("Predicted intent:", intent_inv_map[pred_intent.item()])
print("Predicted sentiment:", sentiment_inv_map[pred_sentiment.item()])


Predicted intent: Loan
Predicted sentiment: Urgent


In [27]:
test_text = "my loan topup request has been pending for 7 weeks"
enc = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)
outputs = encoder(**enc)
cls_repr = outputs.last_hidden_state[:, 0]
pred_intent = torch.argmax(intent_head(cls_repr), dim=1)
pred_sentiment = torch.argmax(sentiment_head(cls_repr), dim=1)

# Reverse mapping for readability
intent_inv_map = {v: k for k, v in intent_map.items()}
sentiment_inv_map = {v: k for k, v in sentiment_map.items()}

print("Predicted intent:", intent_inv_map[pred_intent.item()])
print("Predicted sentiment:", sentiment_inv_map[pred_sentiment.item()])


Predicted intent: Loan
Predicted sentiment: Urgent


Frequently asked questions which can be used for NLP part



"Why is my account restricted? I submitted KYC documents last week." KYC ANGRY

"my loan topup request has been pending for 7 weeks" LOAN ANXIOUS

"I'm locked out of my net banking. How do I regain access?" Account access  ANXIOUS

"My Aadhaar name and bank name do not match. How do I fix this"  KYC CONFUSED

"Multiple small transactions are showing up that I did not make"   Fraud/Unauthorized   Neutral
